# 22. Model Validation Against Empirical Frequencies

This notebook validates that the models trained in notebook 21 (Test 4) can accurately predict empirical edge frequencies.

## Objectives:
1. Load empirical frequencies for CbG edge type
2. Load trained models from Test 4 (Adam NN, L-BFGS NN, LogReg, SimpleNN)
3. Make predictions using the explicit method from notebook 04 (to avoid any sigmoid ambiguity)
4. Compare predicted vs empirical frequencies
5. Assess model calibration and identify systematic biases

## Key Insight:
Binary classification accuracy (AUC) measures ranking ability, but correlation with empirical frequencies measures calibration.
A model can have high AUC but poor calibration if predictions are systematically biased.


In [ ]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

# Add src to path - try both relative paths
sys.path.insert(0, '../src')
sys.path.insert(0, 'src')

# Import model classes
from simple_models import SingleLayerNN
from model_comparison import SimpleNN

# Create results directory
results_dir = '../results/empirical_frequency_validation'
os.makedirs(results_dir, exist_ok=True)

print("Imports complete")
print(f"Results will be saved to: {results_dir}")

In [ ]:
# Load empirical frequencies for CbG edge type
print("Loading empirical frequencies...")

# Try multiple possible paths
possible_paths = [
    '../results/empirical_edge_frequencies/edge_frequency_by_degree_CbG.csv',
    'results/empirical_edge_frequencies/edge_frequency_by_degree_CbG.csv'
]

empirical_df = None
for path in possible_paths:
    if os.path.exists(path):
        empirical_df = pd.read_csv(path)
        print(f"Loaded from: {path}")
        break

if empirical_df is None:
    raise FileNotFoundError(f"Could not find empirical frequencies file in any of: {possible_paths}")

print(f"\nLoaded {len(empirical_df):,} unique (source_degree, target_degree) pairs")
print(f"\nFirst few rows:")
print(empirical_df.head())

print(f"\nEmpirical frequency statistics:")
print(empirical_df['frequency'].describe())

print(f"\nDegree ranges:")
print(f"  Source degrees: {empirical_df['source_degree'].min()} - {empirical_df['source_degree'].max()}")
print(f"  Target degrees: {empirical_df['target_degree'].min()} - {empirical_df['target_degree'].max()}")

In [ ]:
# Load all Test 5 trained models (class weighting, NO StandardScaler)
print("Loading Test 5 models...\n")

# Try multiple possible paths
possible_dirs = ['results/nn_optimizer_comparison', '../results/nn_optimizer_comparison', '/Users/gillenlu/Library/CloudStorage/OneDrive-TheUniversityofColoradoDenver/Repositories/Context-Aware-Path-Probability/results/nn_optimizer_comparison']

model_dir = None
for path in possible_dirs:
    if os.path.exists(path):
        model_dir = path
        print(f"Using model directory: {model_dir}")
        break

if model_dir is None:
    raise FileNotFoundError(f"Could not find model directory in any of: {possible_dirs}")

# Load Adam NN
with open(f'{model_dir}/test5_single_layer_nn_adam_unweighted_scaled.pkl', 'rb') as f:
    adam_results = pickle.load(f)
model_adam = adam_results['model']
# print(f"Loaded Adam NN (AUC: {adam_results['metrics']['auc']:.4f})") # Metrics not saved in Test 5 models

# Load L-BFGS NN
with open(f'{model_dir}/test5_single_layer_nn_lbfgs_unweighted_scaled.pkl', 'rb') as f:
    lbfgs_results = pickle.load(f)
model_lbfgs = lbfgs_results['model']
# print(f"Loaded L-BFGS NN (AUC: {lbfgs_results['metrics']['auc']:.4f})") # Metrics not saved in Test 5 models

# Load LogReg
with open(f'{model_dir}/test5_logreg_unweighted_scaled.pkl', 'rb') as f:
    logreg_results = pickle.load(f)
model_logreg = logreg_results['model']
# print(f"Loaded LogReg (AUC: {logreg_results['metrics']['auc']:.4f})") # Metrics not saved in Test 5 models

# Load SimpleNN
with open(f'{model_dir}/test5_simple_nn_unweighted_scaled.pkl', 'rb') as f:
    simplenn_results = pickle.load(f)
model_simplenn = simplenn_results['model']
# print(f"Loaded SimpleNN (AUC: {simplenn_results['metrics']['auc']:.4f})") # Metrics not saved in Test 5 models

print(f"\nAll Test 5 models loaded successfully")
print(f"Test 5: Class weighting enabled, NO StandardScaler")

In [ ]:
# Prepare features for prediction (NO scaling - Test 3 models trained on raw features)
print("Preparing features...\n")

# Extract degree pairs from empirical data
X = empirical_df[['source_degree', 'target_degree']].values
y_empirical = empirical_df['frequency'].values

print(f"Feature matrix shape: {X.shape}")
print(f"\nFeatures (raw degrees - NO scaling):")
print(f"  Source degrees: min={X[:, 0].min():.1f}, max={X[:, 0].max():.1f}, mean={X[:, 0].mean():.1f}")
print(f"  Target degrees: min={X[:, 1].min():.1f}, max={X[:, 1].max():.1f}, mean={X[:, 1].mean():.1f}")
print(f"\nUsing raw degree features to match Test 3 training data")

In [ ]:
# Make predictions with Adam NN (using explicit notebook 04 method)
print("Making predictions with Adam NN...")

model_adam.eval()
with torch.no_grad():
    X_tensor = torch.FloatTensor(X)
    # Use explicit method from notebook 04 - bypass forward() to avoid any sigmoid ambiguity
    raw_logits = model_adam.network(X_tensor).squeeze()
    pred_adam = torch.sigmoid(raw_logits).numpy()

print(f"Predictions shape: {pred_adam.shape}")
print(f"Prediction range: [{pred_adam.min():.4f}, {pred_adam.max():.4f}]")
print(f"Prediction mean: {pred_adam.mean():.4f}")
print(f"Prediction std: {pred_adam.std():.4f}")

In [ ]:
# Make predictions with L-BFGS NN (using explicit notebook 04 method)
print("Making predictions with L-BFGS NN...")

model_lbfgs.eval()
with torch.no_grad():
    X_tensor = torch.FloatTensor(X)
    # Use explicit method from notebook 04
    raw_logits = model_lbfgs.network(X_tensor).squeeze()
    pred_lbfgs = torch.sigmoid(raw_logits).numpy()

print(f"Predictions shape: {pred_lbfgs.shape}")
print(f"Prediction range: [{pred_lbfgs.min():.4f}, {pred_lbfgs.max():.4f}]")
print(f"Prediction mean: {pred_lbfgs.mean():.4f}")
print(f"Prediction std: {pred_lbfgs.std():.4f}")

In [ ]:
# Make predictions with Logistic Regression
print("Making predictions with Logistic Regression...")

pred_logreg = model_logreg.predict_proba(X)[:, 1]

print(f"Predictions shape: {pred_logreg.shape}")
print(f"Prediction range: [{pred_logreg.min():.4f}, {pred_logreg.max():.4f}]")
print(f"Prediction mean: {pred_logreg.mean():.4f}")
print(f"Prediction std: {pred_logreg.std():.4f}")

In [ ]:
# Make predictions with SimpleNN (using explicit notebook 04 method)
print("Making predictions with SimpleNN...")

model_simplenn.eval()
with torch.no_grad():
    X_tensor = torch.FloatTensor(X)
    # Use explicit method from notebook 04
    raw_logits = model_simplenn.network(X_tensor).squeeze()
    pred_simplenn = torch.sigmoid(raw_logits).numpy()

print(f"Predictions shape: {pred_simplenn.shape}")
print(f"Prediction range: [{pred_simplenn.min():.4f}, {pred_simplenn.max():.4f}]")
print(f"Prediction mean: {pred_simplenn.mean():.4f}")
print(f"Prediction std: {pred_simplenn.std():.4f}")

In [ ]:
# Calculate correlations and metrics for all models
print("Calculating correlations and metrics...\n")

models_data = [
    ('Adam NN', pred_adam),
    ('L-BFGS NN', pred_lbfgs),
    ('LogReg', pred_logreg),
    ('SimpleNN', pred_simplenn)
]

results = []

for name, predictions in models_data:
    # Pearson correlation
    r_pearson, p_pearson = pearsonr(y_empirical, predictions)
    
    # Spearman correlation
    r_spearman, p_spearman = spearmanr(y_empirical, predictions)
    
    # RMSE
    rmse = np.sqrt(mean_squared_error(y_empirical, predictions))
    
    # MAE
    mae = mean_absolute_error(y_empirical, predictions)
    
    # R-squared
    r2 = r2_score(y_empirical, predictions)
    
    results.append({
        'Model': name,
        'Pearson_r': r_pearson,
        'Spearman_r': r_spearman,
        'R2': r2,
        'RMSE': rmse,
        'MAE': mae
    })
    
    print(f"{name}:")
    print(f"  Pearson r: {r_pearson:.4f} (p={p_pearson:.2e})")
    print(f"  Spearman r: {r_spearman:.4f}")
    print(f"  R²: {r2:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print()

# Create comparison dataframe
comparison_df = pd.DataFrame(results)
print("\nComparison Table:")
print(comparison_df.to_string(index=False))


In [ ]:
# Scatter plot: Adam NN predictions vs empirical frequencies
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# Get metrics for annotation
r_pearson, _ = pearsonr(y_empirical, pred_adam)
rmse = np.sqrt(mean_squared_error(y_empirical, pred_adam))

# Scatter plot
ax.scatter(y_empirical, pred_adam, alpha=0.5, s=20)

# Diagonal reference line
ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect prediction')

# Labels and title
ax.set_xlabel('Empirical Frequency', fontsize=12)
ax.set_ylabel('Predicted Frequency', fontsize=12)
ax.set_title('Adam NN: Predicted vs Empirical Frequencies', fontsize=14, fontweight='bold')

# Annotation
ax.text(0.05, 0.95, f'Pearson r = {r_pearson:.4f}\nRMSE = {rmse:.4f}',
        transform=ax.transAxes, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(f'{results_dir}/adam_nn_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/adam_nn_scatter.png")


In [ ]:
# Scatter plot: L-BFGS NN predictions vs empirical frequencies
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

r_pearson, _ = pearsonr(y_empirical, pred_lbfgs)
rmse = np.sqrt(mean_squared_error(y_empirical, pred_lbfgs))

ax.scatter(y_empirical, pred_lbfgs, alpha=0.5, s=20)
ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect prediction')

ax.set_xlabel('Empirical Frequency', fontsize=12)
ax.set_ylabel('Predicted Frequency', fontsize=12)
ax.set_title('L-BFGS NN: Predicted vs Empirical Frequencies', fontsize=14, fontweight='bold')

ax.text(0.05, 0.95, f'Pearson r = {r_pearson:.4f}\nRMSE = {rmse:.4f}',
        transform=ax.transAxes, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(f'{results_dir}/lbfgs_nn_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/lbfgs_nn_scatter.png")


In [ ]:
# Scatter plot: LogReg predictions vs empirical frequencies
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

r_pearson, _ = pearsonr(y_empirical, pred_logreg)
rmse = np.sqrt(mean_squared_error(y_empirical, pred_logreg))

ax.scatter(y_empirical, pred_logreg, alpha=0.5, s=20)
ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect prediction')

ax.set_xlabel('Empirical Frequency', fontsize=12)
ax.set_ylabel('Predicted Frequency', fontsize=12)
ax.set_title('Logistic Regression: Predicted vs Empirical Frequencies', fontsize=14, fontweight='bold')

ax.text(0.05, 0.95, f'Pearson r = {r_pearson:.4f}\nRMSE = {rmse:.4f}',
        transform=ax.transAxes, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(f'{results_dir}/logreg_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/logreg_scatter.png")


In [ ]:
# Scatter plot: SimpleNN predictions vs empirical frequencies
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

r_pearson, _ = pearsonr(y_empirical, pred_simplenn)
rmse = np.sqrt(mean_squared_error(y_empirical, pred_simplenn))

ax.scatter(y_empirical, pred_simplenn, alpha=0.5, s=20)
ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect prediction')

ax.set_xlabel('Empirical Frequency', fontsize=12)
ax.set_ylabel('Predicted Frequency', fontsize=12)
ax.set_title('SimpleNN: Predicted vs Empirical Frequencies', fontsize=14, fontweight='bold')

ax.text(0.05, 0.95, f'Pearson r = {r_pearson:.4f}\nRMSE = {rmse:.4f}',
        transform=ax.transAxes, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(f'{results_dir}/simplenn_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/simplenn_scatter.png")


In [ ]:
# Combined 2x2 comparison plot
fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes = axes.flatten()

for idx, (name, predictions) in enumerate(models_data):
    ax = axes[idx]
    
    r_pearson, _ = pearsonr(y_empirical, predictions)
    rmse = np.sqrt(mean_squared_error(y_empirical, predictions))
    
    ax.scatter(y_empirical, predictions, alpha=0.5, s=10)
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2)
    
    ax.set_xlabel('Empirical Frequency', fontsize=11)
    ax.set_ylabel('Predicted Frequency', fontsize=11)
    ax.set_title(name, fontsize=13, fontweight='bold')
    
    ax.text(0.05, 0.95, f'r = {r_pearson:.3f}\nRMSE = {rmse:.3f}',
            transform=ax.transAxes, verticalalignment='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

fig.suptitle('Model Comparison: Predicted vs Empirical Frequencies',
             fontsize=16, fontweight='bold', y=0.995)

plt.tight_layout()
plt.savefig(f'{results_dir}/all_models_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/all_models_comparison.png")


In [ ]:
# Residual analysis for all models
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, (name, predictions) in enumerate(models_data):
    ax = axes[idx]
    
    residuals = predictions - y_empirical
    
    ax.scatter(y_empirical, residuals, alpha=0.5, s=10)
    ax.axhline(y=0, color='r', linestyle='--', linewidth=2)
    
    ax.set_xlabel('Empirical Frequency', fontsize=11)
    ax.set_ylabel('Residual (Predicted - Empirical)', fontsize=11)
    ax.set_title(f'{name}: Residual Plot', fontsize=13, fontweight='bold')
    
    # Add statistics
    mean_resid = residuals.mean()
    std_resid = residuals.std()
    ax.text(0.05, 0.95, f'Mean = {mean_resid:.4f}\nStd = {std_resid:.4f}',
            transform=ax.transAxes, verticalalignment='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])

fig.suptitle('Residual Analysis: Checking for Systematic Bias',
             fontsize=16, fontweight='bold', y=0.995)

plt.tight_layout()
plt.savefig(f'{results_dir}/residual_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/residual_analysis.png")


In [ ]:
# Check prediction distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Empirical distribution
axes[0, 0].hist(y_empirical, bins=50, alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Frequency')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Empirical Frequencies', fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Model predictions
plot_positions = [(0, 1), (0, 2), (1, 0), (1, 1)]
for (name, predictions), (i, j) in zip(models_data, plot_positions):
    axes[i, j].hist(predictions, bins=50, alpha=0.7, edgecolor='black', color='orange')
    axes[i, j].set_xlabel('Frequency')
    axes[i, j].set_ylabel('Count')
    axes[i, j].set_title(f'{name} Predictions', fontweight='bold')
    axes[i, j].grid(True, alpha=0.3)
    
    # Add range annotation
    axes[i, j].text(0.95, 0.95, f'Range: [{predictions.min():.3f}, {predictions.max():.3f}]',
                    transform=axes[i, j].transAxes, ha='right', va='top', fontsize=9,
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Remove empty subplot
fig.delaxes(axes[1, 2])

fig.suptitle('Distribution Analysis: Check for Double Sigmoid Compression',
             fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{results_dir}/prediction_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved: {results_dir}/prediction_distributions.png")
print("\nNote: If double sigmoid was present, predictions would be compressed toward 0.5")


In [ ]:
# Save all predictions and results

# Create predictions dataframe
predictions_df = empirical_df.copy()
predictions_df['pred_adam'] = pred_adam
predictions_df['pred_lbfgs'] = pred_lbfgs
predictions_df['pred_logreg'] = pred_logreg
predictions_df['pred_simplenn'] = pred_simplenn

# Add residuals
predictions_df['resid_adam'] = pred_adam - y_empirical
predictions_df['resid_lbfgs'] = pred_lbfgs - y_empirical
predictions_df['resid_logreg'] = pred_logreg - y_empirical
predictions_df['resid_simplenn'] = pred_simplenn - y_empirical

# Save predictions
predictions_df.to_csv(f'{results_dir}/all_predictions.csv', index=False)
print(f"Saved: {results_dir}/all_predictions.csv")

# Save correlation summary
comparison_df.to_csv(f'{results_dir}/correlation_summary.csv', index=False)
print(f"Saved: {results_dir}/correlation_summary.csv")

print("\nAll results saved successfully!")
print(f"\nResults directory: {results_dir}")
print(f"  - Individual scatter plots")
print(f"  - Combined comparison plot")
print(f"  - Residual analysis plot")
print(f"  - Prediction distributions plot")
print(f"  - all_predictions.csv (predictions + residuals)")
print(f"  - correlation_summary.csv (metrics table)")
